# RlLib pour le couloir

Jusqu'à présent nous avons fait usage de StableBaselines qui est très facile à utiliser.
Mais d'autres librairies existent, comme RLlib

## Importations

In [12]:
# LES ALGORITHMES
# Pour StableBaselines
from QLearning import *
from DQN import *
from PPO import *
from helperPlot import *

# Pour RLLib
from ray.rllib.algorithms.dqn import DQNConfig
from ray.rllib.connectors.env_to_module import FlattenObservations
from ray import train, tune

# Aide
import time
from pprint import pprint

# POUR AFFICHAGE
import matplotlib.pyplot as plt
import seaborn as sns

# POUR ENVIRONNEMENT
import gymnasium
from gymnasium import spaces

## Environnement

In [62]:
class LineWorldEnv(gymnasium.Env):
    def __init__(self, options = None):
        super(LineWorldEnv, self).__init__()
        n_states = options['n_states']
        self.n_states = n_states
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Discrete(n_states)
        self.state = 0

    def reset(self, seed = None, options = None):
        super().reset(seed = seed)
        self.state = 0
        return self.state, {}

    def step(self, action):
        if action == 0 and self.state > 0:
            self.state -= 1
        elif action == 1 and self.state < self.n_states - 1:
            self.state += 1
        done = self.state == self.n_states - 1
        reward = 10 if done else -1
        return self.state, reward, done, False, {}

    def render(self):
        print("État actuel :", self.state)

## Essai RLLib

In [79]:
# Configure the algorithm.
config = (
    DQNConfig()
    .environment(
        LineWorldEnv,
        env_config = {'n_states' : 3},
    )
    .env_runners(
        # Observations are discrete (ints) -> We need to flatten (one-hot) them.
        env_to_module_connector=lambda env: FlattenObservations(),
    )
)


algo = config.build_algo()


for _ in range(2):
    algo.train()


module = algo.get_module()

2025-06-13 11:20:08,283	WARNING util.py:61 -- Install gputil for GPU system monitoring.


In [80]:
def one_hot(config, state) :
    try :
        n = sum(config)
        obs = [0 for _ in range(n)]
        obs[int(state[0])] = 1
        k=0
        for i in range(1, len(config)) :
            k += config[i-1]
            obs[k+int(state[i])] = 1
    except TypeError :
        obs = [0 for _ in range(config)]
        obs[int(state)] = 1
    
    return np.array(obs, dtype = np.float32)


size = 3

# Create the RL environment to test against (same as was used for training earlier).
env = LineWorldEnv(options = {'n_states' : size})

episode_return = 0.0
done = False

# Reset the env to get the initial observation.
obs, info = env.reset()

while not done:
    env.render()

    # Compute the next action from a batch (B=1) of observations.
    obs_batch = th.from_numpy(one_hot(size, obs)).unsqueeze(0)  # add batch B=1 dimension
    model_outputs = module.forward_inference({'obs': obs_batch})

    # Extract the action distribution parameters from the output and dissolve batch dim.
    action_dist_params = model_outputs["actions"][0].numpy()
    
    greedy_action = action_dist_params

    # Send the action to the environment for the next step.
    obs, reward, terminated, truncated, info = env.step(greedy_action)

    # Perform env-loop bookkeeping.
    episode_return += reward
    done = terminated or truncated
env.render()
print(f"Reached episode return of {episode_return}.")

État actuel : 0
État actuel : 1
État actuel : 2
Reached episode return of 9.0.
